In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import cosine_distances
from sklearn.cluster import SpectralClustering
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import silhouette_score
from collections import defaultdict
from scipy.linalg import eigh
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, auc
from rouge_score import rouge_scorer
import networkx as nx
from tqdm import tqdm
import torch
import json
import pandas as pd
import numpy as np
import os
import re
import gc
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import accelerate
print(accelerate.__version__)

In [ ]:
# Load SQuAD data (train set)
squad_train_path = ".../train-v1.1.json"
with open(squad_train_path, "r") as f:
    squad_data = json.load(f)

# Flatten into question-answer list
qa_list = []
for article in squad_data["data"]:
    for para in article["paragraphs"]:
        context = para["context"]
        for qa in para["qas"]:
            question = qa["question"]
            # Use the first answer (there may be multiple valid ones)
            answer_text = qa["answers"][0]["text"] if qa["answers"] else None
            qa_list.append({
                "context": context,
                "question": question,
                "answer_main": answer_text
            })

df_squad = pd.DataFrame(qa_list)


In [ ]:
model_id = 'meta-llama/Llama-2-7b-chat-hf'
hf_token = '...'  
cache_dir = '...'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=hf_token,
    cache_dir=cache_dir
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# --- Load model ---
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=hf_token,
    cache_dir=cache_dir,
    torch_dtype=dtype
)
model.to(device)

e5_model = SentenceTransformer("intfloat/e5-large-v2", device="cpu") 
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [ ]:
from scipy import special


def _A_p_stable(kappa, d):
    """Stable A_p(kappa) = I_{d/2}(k) / I_{d/2-1}(k) with asymptotics."""
    if kappa <= 1e-8:              # small-kappa series
        return kappa / d
    if kappa >= 1e4:               # large-kappa asymptotic (2 terms)
        return 1.0 - (d-1)/(2.0*kappa) + (d-1)*(d-3)/(4.0*kappa**2)
    num = special.ive(d/2.0, kappa)
    den = special.ive(d/2.0 - 1.0, kappa)
    return num / (den + 1e-300)

def _dA_p_from_A(kappa, A, d):
    if kappa <= 0:
        return 1.0 - A*A
    return 1.0 - A*A - ((d-1)/kappa)*A

def _kappa_initial(Rbar, d):
    Rbar = float(np.clip(Rbar, 1e-12, 1 - 1e-12))
    return max((Rbar * (d - Rbar**2)) / (1.0 - Rbar**2), 1e-6)

def _kappa_upper_bound(Rbar, d):
    Rbar = float(np.clip(Rbar, 1e-9, 1 - 1e-9))
    return max((d-1) / (2.0 * (1.0 - Rbar)), 10.0)

def vmf_stats(embeddings, newton_iters=8, tol=1e-7, use_bisection_backup=True):
    X = np.asarray(embeddings, dtype=np.float64)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    X = X / norms

    N, d = X.shape
    R = X.sum(axis=0)
    R_norm = float(np.linalg.norm(R))
    if R_norm < 1e-12:
        return {
            "mu": np.zeros(d),
            "kappa": 0.0,
            "A_p": 0.0,
            "var_parallel": 1.0/d,
            "var_perp": 1.0/d,
            "trace_var": 1.0
        }

    mu_hat = R / R_norm
    Rbar = R_norm / max(N, 1)

    kappa = _kappa_initial(Rbar, d)
    ok_newton = True
    for _ in range(newton_iters):
        A = _A_p_stable(kappa, d)
        f = A - Rbar
        if abs(f) < tol:
            break
        dA = _dA_p_from_A(kappa, A, d)
        step = f / (dA + 1e-18)
        kappa_new = kappa - step
        if not np.isfinite(kappa_new) or kappa_new <= 0 or abs(step) > 0.5*max(1.0, kappa):
            ok_newton = False
            break
        kappa = kappa_new

    if (not ok_newton) and use_bisection_backup:
        lo = 0.0
        hi = _kappa_upper_bound(Rbar, d)
        for _ in range(60):
            mid = 0.5*(lo + hi)
            A_mid = _A_p_stable(mid, d)
            if A_mid < Rbar:
                lo = mid
            else:
                hi = mid
            if (hi - lo) < 1e-7 * (1.0 + mid):
                break
        kappa = 0.5*(lo + hi)

    A = float(Rbar)
    var_perp = A / (kappa + 1e-18)
    var_parallel = 1.0 - ((d - 1.0) * A / (kappa + 1e-18)) - A*A
    trace_var = 1.0 - A*A

    var_perp = float(max(var_perp, 0.0))
    var_parallel = float(max(var_parallel, 0.0))
    trace_var = float(max(trace_var, 0.0))

    return {
        "mu": mu_hat,
        "kappa": float(kappa),
        "A_p": A,
        "var_parallel": var_parallel,
        "var_perp": var_perp,
        "trace_var": trace_var
    }

def format_prompt(question, context=None):
    if context:
        return (
            "[INST] Read the passage and answer the question.\n"
            f"Passage: {context}\nQuestion: {question} [/INST]"
        )
    return f"[INST] Answer the following question.\nQuestion: {question} [/INST]"


def normalize_generation(text):
    text = text.strip().lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text
    
def generate_completions(prompt, n=10, batch_size=2):
    generations = []

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(prompt, return_tensors="pt", padding=True)
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    for start in range(0, n, batch_size):
        bs = min(batch_size, n - start)
        with torch.no_grad():
            output_ids = model.generate(
                input_ids=input_ids.repeat(bs, 1),
                attention_mask=attention_mask.repeat(bs, 1),
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                do_sample=True,
                num_return_sequences=bs,
                pad_token_id=tokenizer.pad_token_id
            )

        for i in range(bs):
            gen_ids = output_ids[i][input_ids.shape[1]:]
            generations.append(tokenizer.decode(gen_ids, skip_special_tokens=True).strip())

        torch.cuda.empty_cache()
        gc.collect()

    return generations


def check_entailment(premise, hypothesis):
    inputs = nli_tokenizer(premise, hypothesis, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        logits = nli_model(**inputs).logits
    probs = F.softmax(logits, dim=1)
    return probs[0][2].item()  # entailment score

def deberta_cluster(gens, threshold=0.8):
    
    n = len(gens)
    G = nx.Graph()
    G.add_nodes_from(range(n))  # add all nodes, including isolated ones

    # Build edges based on entailment threshold
    for i in range(n):
        for j in range(i + 1, n):
            ent1 = check_entailment(gens[i], gens[j])
            ent2 = check_entailment(gens[j], gens[i])
            if ent1 > threshold and ent2 > threshold:
                G.add_edge(i, j)

    # Connected components will include singletons automatically
    clusters = list(nx.connected_components(G))

    # Assign cluster ids
    labels = np.zeros(n, dtype=int)
    for cluster_id, nodes in enumerate(clusters):
        for node in nodes:
            labels[node] = cluster_id

    return labels

# -----------------------------
# Entropy Computation
# -----------------------------
import numpy as np

def cluster_assignment_entropy(semantic_ids):
    counts = np.bincount(semantic_ids)
    total = counts.sum()
    if total == 0:
        return 0.0
    probs = counts.astype(np.float64) / total
    probs = np.clip(probs, 1e-12, 1.0)  # avoid log(0)
    H = -np.sum(probs * np.log(probs))
    return float(max(H, 0.0))  # clamp to 0 if tiny negative

# -----------------------------
# Apply DeBERTa Entropy
# -----------------------------
def deberta_entropy(row):
    gens = row["generations"]
    if not gens:
        return None
    try:
        labels = deberta_cluster(gens)
        return cluster_assignment_entropy(labels)
    except Exception as e:
        print(f"[Error] {e}")
        return None


In [ ]:
results = []
MAX_NEW_TOKENS = 64
N_GENERATIONS = 10
TEMPERATURE = 0.7
TOP_P = 0.9

df_squad_sampled = df_squad.sample(n=300, random_state=42)

for idx, row in tqdm(df_squad_sampled.iterrows(), total=len(df_squad_sampled)):
    question = row["question"]
    #prompt = format_prompt(question, row["context"])
    prompt = format_prompt(question, None)
   
   
    generations = generate_completions(prompt, n=N_GENERATIONS)
    cleaned = [f"passage: {normalize_generation(t)}" for t in generations if t.strip()]
    embeddings = e5_model.encode(cleaned, convert_to_tensor=True, normalize_embeddings=True).cpu().numpy()

    stats = vmf_stats(embeddings)

    results.append({
        "question": question,
        "reference_answer": row["answer_main"],
        "generations": generations,
        "kappa": stats["kappa"],
        "trace_var": stats["trace_var"],
        "var_parallel": stats["var_parallel"],
        "var_perp": stats["var_perp"]
    })

df_results = pd.DataFrame(results)
nli_tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-large-mnli")
nli_model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-large-mnli").to(device)
nli_model.eval()
df_results["semantic_entropy"] = df_results.apply(deberta_entropy, axis=1)
multi_sent_path = "vms_deberta_squad_300_multi_sent.csv"
df_results.to_csv(multi_sent_path, index=False)

In [ ]:
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def is_correct_rouge(row, threshold=0.1):
    generations = row["generations"]
    ref = row["reference_answer"]

    # Handle missing data
    if not isinstance(ref, str) or not isinstance(generations, list) or len(generations) == 0:
        return False

    top1 = generations[0]
    if not isinstance(top1, str):
        return False

    score = scorer.score(ref.lower(), top1.lower())
    rouge_l_f1 = score['rougeL'].fmeasure
    return rouge_l_f1 >= threshold

df_results["is_correct"] = df_results.apply(is_correct_rouge, axis=1)
df_results["y_true"] = (~df_results["is_correct"]).astype(int)  # 1 = incorrect, 0 = correct

In [ ]:
y_true = df_results["y_true"] 

auroc_kappa = roc_auc_score(y_true, 1/df_results["kappa"])
auroc_deberta = roc_auc_score(y_true, df_results["semantic_entropy"])

print(f"AUROC (kappa):           {auroc_kappa:.3f}")
print(f"AUROC (deberta_entropy): {auroc_deberta:.3f}")

In [ ]:
round((1/df_results['semantic_entropy']).mean(), 5)